# HW3: Image Classification and Explainable AI with CNNs and Transformers

## Overview

In this homework, you will use a CNN and a Transformer for image classification, then apply explainable AI techniques to connect each model’s predictions to the image regions that matter most.

You will work with two pre-trained backbones:

1. ResNet-50 (CNN)
2. DINOv2 (Vision Transformer)

### Part 1: Linear Probing on CUB-200-2011

*CUB-200-2011 Dataset link: https://www.vision.caltech.edu/datasets/cub_200_2011/*

Use each pre-trained model as a frozen feature extractor, then train a linear classifier (linear probing) on top of the extracted features. Train on the CUB-200-2011 training split, then (1) report each model’s classification accuracy on the CUB test split, and (2) visualize the class similarity matrix based on the trained linear classifier.

### Part 2: Explainability for the Predictions

Implement the following techniques to analyze what the models have learned:

1. Grad-CAM (https://arxiv.org/abs/1610.02391): Gradient-based class activation maps w.r.t. the ground truth label and the predicted label

2. Finer-CAM (https://arxiv.org/abs/2501.11309): Fine-grained class activation maps that highlight the differences between visually similar classes. We will use one reference with the second largest label probability to compare the activation differences to the ground truth label and the predicted label   

# Part 1: Linear Probing on CUB-200-2011

In Part 1, we will

1. Download the CUB Dataset

2. Create the PyTorch Dataset class, and train/test data loader

3. **Linear Probing**: Train a linear classifier upon the features from the pretained ResNet-50 and Dino-v2 models on the CUB-train split, report the CUB-test split accruacy and visualize the class similarity map
   - *Your implmentations happen here*
   - *See details in the following section*

## 1. Download the CUB Dataset

In [ ]:
import os
import shutil

print("Cleaning up files and directories if existed...")
# Remove any existing tarballs and folders
for path in ['/content/CUB_200_2011.tgz', '/content/CUB_200_2011.tgz.1', '/content/CUB_200_2011']:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

print("Downloading a fresh copy of the CUB_200_2011 dataset...")
!wget -O /content/CUB_200_2011.tgz https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz

print("Extracting the newly downloaded archive...")
!tar -xzf /content/CUB_200_2011.tgz -C /content/

# Verification
required_files = ["images.txt", "image_class_labels.txt", "train_test_split.txt", "bounding_boxes.txt", "classes.txt"]
missing = [f for f in required_files if not os.path.exists(os.path.join("/content/CUB_200_2011", f))]

if not missing:
    print("Success! All dataset metadata files are successfully restored:")
    !ls -la /content/CUB_200_2011
else:
    print(f"Error: missing files: {missing}")

## 2. Create the PyTorch Dataset class, and train/test data loader

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import DataLoader

class CUBDataset(Dataset):
    def __init__(self, dataset_dir="/content/CUB_200_2011", train=True, transform=None, crop_to_bbox=False):
        """
        Custom Dataset for CUB-200-2011.

        Args:
            dataset_dir (str): Path to the root 'CUB_200_2011' directory.
            train (bool): If True, loads the training split; otherwise loads the test split.
            transform (callable, optional): PyTorch transforms to apply to the images.
            crop_to_bbox (bool): If True, crops the image to the bird's bounding box before transforming.
        """
        self.dataset_dir = dataset_dir
        self.transform = transform
        self.crop_to_bbox = crop_to_bbox

        # Define paths to metadata files
        images_txt = os.path.join(dataset_dir, "images.txt")
        labels_txt = os.path.join(dataset_dir, "image_class_labels.txt")
        split_txt = os.path.join(dataset_dir, "train_test_split.txt")
        bboxes_txt = os.path.join(dataset_dir, "bounding_boxes.txt")
        classes_txt = os.path.join(dataset_dir, "classes.txt")

        # Load metadata using pandas
        images_df = pd.read_csv(images_txt, sep=r'\s+', names=["img_id", "filepath"])
        labels_df = pd.read_csv(labels_txt, sep=r'\s+', names=["img_id", "label"])
        split_df = pd.read_csv(split_txt, sep=r'\s+', names=["img_id", "is_train"])
        bboxes_df = pd.read_csv(bboxes_txt, sep=r'\s+', names=["img_id", "x", "y", "width", "height"])
        classes_df = pd.read_csv(classes_txt, sep=r'\s+', names=["label", "class_name"])

        # Merge all metadata DataFrames together
        metadata = images_df.merge(labels_df, on="img_id")
        metadata = metadata.merge(split_df, on="img_id")
        metadata = metadata.merge(bboxes_df, on="img_id")
        metadata = metadata.merge(classes_df, on="label")

        # Filter train vs test split
        split_val = 1 if train else 0
        self.data = metadata[metadata["is_train"] == split_val].reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Load the image
        img_path = os.path.join(self.dataset_dir, "images", row["filepath"])
        image = Image.open(img_path).convert("RGB")

        # Crop to bounding box if requested
        if self.crop_to_bbox:
            # Bounding box is defined as (x, y, width, height)
            x, y, w, h = row["x"], row["y"], row["width"], row["height"]
            image = image.crop((x, y, x + w, y + h))

        # Target label (convert 1-indexed CUB class labels to 0-indexed)
        label = int(row["label"]) - 1

        # Class name
        class_name = row["class_name"]

        # Apply torchvision transforms if provided
        if self.transform:
            image = self.transform(image)

        return image, label, class_name


# Define Standard ImageNet Transforms
# ResNet-50 expects 224x224 images normalized with ImageNet stats
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Instantiate Datasets and Dataloaders
train_dataset = CUBDataset(dataset_dir="/content/CUB_200_2011", train=True, transform=train_transform, crop_to_bbox=True)
test_dataset = CUBDataset(dataset_dir="/content/CUB_200_2011", train=False, transform=test_transform, crop_to_bbox=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Loaded {len(train_dataset)} training images.")
print(f"Loaded {len(test_dataset)} testing images.")

## 3. **Linear Probing**

Train a linear classifier upon the features from the pretained ResNet-50 and Dino-v2 models on the CUB-train split, report the CUB-test split accruacy and visualize the class similarity map
   - *You should fill in every secion enclosed by*
   
   "#### Your job \<number\> starts here: ####"
   
   "#### Your job \<number\> ends here: ####"


In [ ]:
"""
1. Define ResNet-50 + linear classifier model, and make sure the ResNet-50 weights are frozen
including the batch norm layers
"""

from torchvision.models import resnet50, ResNet50_Weights
import torch.nn as nn

class ResNet50LinearProbe(nn.Module):
    def __init__(self, num_classes=200, weights=ResNet50_Weights.IMAGENET1K_V2):
        super().__init__()
        self.backbone = resnet50(weights=weights) # Initiate and load pretrained ResNet-50 weights
        self.backbone.fc = nn.Identity() # Replace the original 1000-class fc layer to
        # identity layer in order to output 2048-d pooled feature

        for p in self.backbone.parameters():
            # Freeze the backbone parameters
            p.requires_grad = False

        self.fc = nn.Linear(2048, num_classes, bias=False) # default bias is True. Set to False for
        # simplicity

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval() # keeps Batch Norm running stats fixed
        return self

    def forward(self, x):
        with torch.no_grad():
            # do not compute the gradients for the frozen backbone to save memory and compute
            feats = self.backbone(x) # already flattened to (Batch size, 2048)
        return self.fc(feats)



In [ ]:
"""
2. Training and Evaluation Loops
"""
import torch.optim as optim
import time
import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42) # fix the random seed, so different runs are comparable

# Set the device to GPU (cuda) to train the linear classifier efficiently
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Instantiate the ResNet-50 + Linear Probing Model
model = ResNet50LinearProbe(num_classes=200, weights=ResNet50_Weights.IMAGENET1K_V2).to(device)


#### Your job 1 starts here: (5 pts) ####
# Which criterion / loss objective should be used to tain the linear classifier defined above
# criterion = ?? # (Uncomment and fill in your answer in this line)

# Set up an AdamW optimizer to update ONLY the weights of the linear classifier,
# and use the learning rate: 1e-3, weight decay: 0.1
# optimizer = ?? # (Uncomment and fill in your answer in this line)
#### Your job 1 ends here: ####



def evaluate(model, dataloader, criterion, device):
    """
    Evaluate the model on the given dataloader.
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        #### Your job 2 starts here: (5 pts) ####
        # Please fill in this section to loop through the dataloader in order to
        # compute the loss and accruacy between the labels and predictions
        # You may consult the following training loop for hints
        #### Your job 2 ends here: ####

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc


# Run training for 5 epochs
num_epochs = 5
train_loss_list_per_epoch = []
train_acc_list_per_epoch = []
test_loss_list_per_epoch = []
test_acc_list_per_epoch = []
print("Starting Linear Probing Training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    start_time = time.time()

    for images, labels, _ in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass & Optimize
        #### Your job 3 starts here: (10 pts) ####
        # Please fill in this section to initize the gradients to zero values,
        # compute the gradients, and update the weights accordinly. Your answer
        # should include three lines.
        #### Your job 3 ends here: ####

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / total
    train_acc = 100.0 * correct / total
    train_loss_list_per_epoch.append(train_loss)
    train_acc_list_per_epoch.append(train_acc)

    # Evaluate on the test dataset
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    test_loss_list_per_epoch.append(test_loss)
    test_acc_list_per_epoch.append(test_acc)
    elapsed = time.time() - start_time

    print(f"Epoch [{epoch+1}/{num_epochs}] ({elapsed:.1f}s) | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")


# Save the trained linear probe model weights
model.eval()
torch.save(model.state_dict(), 'resnet50_linear_probe.pth')
print("Model weights saved successfully to '/content/resnet50_linear_probe.pth'!")
# You can check the saved model from the left-hand side of the Files folder


In [ ]:
"""
3. Plot the training loss and test accrucay for each epoch
"""
import matplotlib.pyplot as plt
#### Your job 4 starts here: (5 pts) ####
# Please use the recorded training loss (train_loss_list_per_epoch), and
# the recorded  test accruacy (test_acc_list_per_epoch) in the above cell
# to plot the training loss w.r.t. epoch curve, and the test accuracy w.r.t. epoch
#### Your job 4 end here: ####

In [ ]:
"""
4. Visualize the class cosine similarity map
"""
import numpy as np

# Load your trained ResNet50LinearProbe model
resenet50_linearprob_model = ResNet50LinearProbe(num_classes=200, weights=None)
resenet50_linearprob_model.load_state_dict(torch.load('resnet50_linear_probe.pth', map_location=device))
resenet50_linearprob_model.to(device).eval()


# Extract the linear classifier weights and save them to numpy array
W = resenet50_linearprob_model.fc.weight.detach().cpu().numpy()   # shape (200, 2048)
np.save("fc_weight.npy", W)
print(W.shape, W.dtype)    # (200, 2048) float32


#### Your job 5 starts here: (5 pts) ####
# Please load the linear classfier weights (fc_weight.npy), compute the cosine similaities
# on 200 classes and visualize the similarity map (200 x 200)
#### Your job 5 ends here: ####

In [ ]:
"""
5. Repeat the above Step 1. to Step 4 but with Dino-V2 model, and use the Average Pooing for fearture extraction
"""

# 5.1 : Load the DinoV2 model and Define Linear Probe Model using pre-trained DINOv2 backbone
dino = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vits14"
).to(device)
print(dino)

class DinoLinearProbe(nn.Module):
    def __init__(self, backbone, embed_dim=384, num_classes=200, feature_extraction_mode="avgpool"):
      # feature_extraction_mode:
      # "cls": Use CLS token features for linear probing,
      # "avgpool": use Average Pooled features from the patch tokens for linear probing ("default")
      super(DinoLinearProbe, self).__init__()
      self.backbone = backbone
      self.feature_extraction_mode = feature_extraction_mode

      # Freeze the backbone weights
      for param in self.backbone.parameters():
          param.requires_grad = False

      self.fc = nn.Linear(embed_dim, num_classes, bias=False)

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()
        return self

    def forward(self, x):
        with torch.no_grad():
            if self.feature_extraction_mode == "cls"
              # Extract the CLS token
              features = self.backbone(x)
            else:
              # Extract intermediate features: "x_norm_patchtokens" represents the normalized patch token features
              features_dict = self.backbone.forward_features(x)
              patch_tokens = features_dict["x_norm_patchtokens"]  # Shape: (Batch, Num_Patches, Embed_Dim)
              # Perform global average pooling over the patch sequence dimension (dim=1)
              pooled_features = patch_tokens.mean(dim=1)          # Shape: (Batch, Embed_Dim)
        out = self.fc(features)
        return out


In [ ]:
#### Your job 6 starts here: (10 pts) ####
# Please use the DinoLinearProbe model class defined above and follow the same steps as applied on ResNet50-linearprobe model,
# complete the linear probing training,
# model saving, plot the CUB training loss and test accruacy, and visualize the 200x200 class similarity map

# 5.2: Training and Evaluation loop

# 5.3: Plot the training loss and test accrucay for each epoch

# 5.4 Visualize the class cosine similarity map

#### Your job 6 end here: ####

# Part 2: Explainability for Predictions

Implement Grad-CAM using the last convolutional feature map (7×7×2048) from layer 4 of ResNet-50. Smooth or upsample the resulting heatmap to the input resolution, then overlay it on the image.

**Grad-CAM** (https://arxiv.org/abs/1610.02391): Gradient-based class activation maps w.r.t. the ground truth label and the predicted label

*Intuitively, Grad-CAM explains either “why did the model predict this” or “where is the evidence for class y”*


In [ ]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

def layer4_map(backbone, x):
    x = backbone.maxpool(backbone.relu(backbone.bn1(backbone.conv1(x))))
    return backbone.layer4(backbone.layer3(backbone.layer2(backbone.layer1(x))))


def grad_cam(model, x, target=None):
    """Grad-CAM on the 2048x7x7 layer4 map.
    Args:
       model: ResNet50-linearProb model
       x: input image (1,3,224,224)
       target: the class label you wanted to compute the class activation map on x (optional)
               If not set, the model's predicted class will be used.
    Returns (cam in [0,1] at input size, target idx).
    """
    model.eval()
    with torch.enable_grad():
        #### Your job 7 starts here: (15 pts) ####
        # (1) Obtain the activation map A, shape (1, 2048, 7, 7), with layer4_map(model.backbone, x).
        # (2) Make sure gradients can flow to A. Hint: the backbone is frozen and x does not require grad,
        #     so A has no autograd graph on its own.
        # (3) Average-pool A over its spatial dims to get p, shape (1, 2048), and make sure p.grad is
        #     available after the backward pass. Hint: p is not a leaf tensor.
        # (4) Compute logits = model.fc(p). Set target to the predicted class if it is None. Then
        #     backpropagate the score of the target class only.
        # After this block, these names must exist: A, p, logits, target
        #### Your job 7 ends here: ####
    assert p.grad is not None, "p.grad is None: make sure p keeps its gradient (retain_grad) and A requires grad"
    assert torch.allclose(p.grad[0], model.fc.weight[target], atol=1e-6), "p.grad should equal the target row of fc.weight"
    alpha = p.grad[:, :, None, None]                 # eq (1): alpha_k = dlogit/dp_k (1/49 scale vanishes after normalizing)
    cam = F.relu((alpha * A.detach()).sum(1))[0].cpu().numpy()   # (7, 7)
    cam = cam - cam.min(); cam = cam / (1e-7 + cam.max())
    cam = cv2.resize(cam, x.shape[-2:][::-1])
    cam = cam - cam.min(); cam = cam / (1e-7 + cam.max())
    return cam, target

# Visualize Grad-CAM on one example
images, labels, class_names = next(iter(test_loader))
x, label = images[0:1].to(device), labels[0].item()

cam, pred = grad_cam(resenet50_linearprob_model, x, target=None)   # target=None -> use predicted class

raw = np.clip(x[0].cpu().numpy().transpose(1, 2, 0) * np.array(std) + np.array(mean), 0, 1)
heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)[..., ::-1] / 255.0   # BGR -> RGB
overlay = 0.4 * heat + 0.6 * raw

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(raw);     ax[0].set_title(f"Original (class {label})"); ax[0].axis("off")
ax[1].imshow(overlay); ax[1].set_title(f"Grad-CAM, target {pred}");   ax[1].axis("off")
plt.show()

In [ ]:
# Visualize Grad-CAM on more examples
n_show, shown = 20, 0 # how many images to visualize

for images, labels, names in test_loader:
    for i in [10, 50]:
        x, label = images[i:i+1].to(device), labels[i].item()
        with torch.no_grad():
            pred = resenet50_linearprob_model(x).argmax(1).item()           # the model's actual prediction
        cam, _ = grad_cam(resenet50_linearprob_model, x, target=pred)      # explains the predicted class

        raw = np.clip(x[0].cpu().numpy().transpose(1, 2, 0) * np.array(std) + np.array(mean), 0, 1)
        heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)[..., ::-1] / 255.0
        overlay = 0.4 * heat + 0.6 * raw

        fig, ax = plt.subplots(1, 2, figsize=(8, 4))
        ax[0].imshow(raw);     ax[0].set_title(f"Original (true {label}, pred {pred})"); ax[0].axis("off")
        ax[1].imshow(overlay); ax[1].set_title(f"Grad-CAM for pred {label}");           ax[1].axis("off")
        plt.show(); plt.close(fig)

        shown += 1
        if shown >= n_show: break
    if shown >= n_show: break

# Finer-CAM

Implement Finer-CAM (https://arxiv.org/pdf/2501.11309) using the last convolutional feature map.

In [ ]:
def finer_cam(model, x, target=None, refs=None, k=1, gamma=0.6):
    """Finer-CAM (paper Eqs. 5-8): Grad-CAM on y^c - gamma * mean_t y^{d_t}.
    refs=None -> the k classes whose logits are closest to the target's logit.
    Args:
       model: ResNet50-linearProb model
       x: input image (1,3,224,224)
       target: the class index c to explain (optional). If None, the model's predicted class is used.
       refs: the reference classes d to compare against (optional). If None, use the k classes whose
             logits are closest to the target's logit (excluding the target itself).
       k: number of reference classes. Please use k=1 for this assignment
       gamma: comparison strength coefficient in Eq. (5) of the paper
    Returns (cam in [0,1] at input size, target idx, list of reference class indices).

    """
    model.eval()
    with torch.enable_grad():
        #### Your job 8 starts here: (15 pts) ####
        # Finer-CAM is Grad-CAM with a different scalar to backpropagate. Instead of the target logit
        # y^c alone, you differentiate the DIFFERENCE  y^c - gamma * mean over references of y^d.
        #
        # (1) Reuse Job 7: compute A (1,2048,7,7), make it a leaf that requires grad, pool it into p
        #     (keep p.grad), and compute logits = model.fc(p), shape (1, 200).
        # (2) Choose the target class c (predicted class if target is None).
        # (3) If refs is None, pick the k reference classes d:
        #       - measure how close every class's logit is to the target's logit (absolute difference),
        #       - sort the classes from closest to farthest,
        #       - take the first k that are NOT the target itself.
        #     Make sure refs ends up as a Python list of ints, even if the caller passed one int.
        # (4) Build the scalar objective  y^c - gamma * mean_t y^{d_t}  from logits[0, ...],
        #     then call backward() on it.
        #
        # After this block these names must exist: A, p, logits, target, refs
        #### Your job 8 ends here: ####
    assert p.grad is not None, "p.grad is None: keep p.grad (retain_grad) and make A require grad"
    expected = model.fc.weight[target] - gamma * model.fc.weight[refs].mean(0)
    assert torch.allclose(p.grad[0], expected, atol=1e-6), "p.grad does not match W[c] - gamma * mean_t W[d_t]"
    alpha = p.grad[:, :, None, None]                      # alpha^c - gamma * mean_t alpha^{d_t}
    cam = F.relu((alpha * A.detach()).sum(1))[0].cpu().numpy()
    cam = cam - cam.min(); cam = cam / (1e-7 + cam.max())
    cam = cv2.resize(cam, x.shape[-2:][::-1])
    cam = cam - cam.min(); cam = cam / (1e-7 + cam.max())
    return cam, target, refs



In [ ]:
# Visualize Finer-CAM on one example
images, labels, class_names = next(iter(test_loader))
x, label = images[0:1].to(device), labels[0].item()

fcam, tgt, refs = finer_cam(resenet50_linearprob_model, x, target=None, k=1, gamma=0.6)

raw = np.clip(x[0].cpu().numpy().transpose(1, 2, 0) * np.array(std) + np.array(mean), 0, 1)
heat = cv2.applyColorMap(np.uint8(255 * fcam), cv2.COLORMAP_JET)[..., ::-1] / 255.0   # BGR -> RGB
overlay = 0.4 * heat + 0.6 * raw

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(raw);     ax[0].set_title(f"Original (true {label}, pred {tgt})"); ax[0].axis("off")
ax[1].imshow(overlay); ax[1].set_title(f"Finer-CAM: {label} vs {refs}", fontsize=9); ax[1].axis("off")
plt.show()

In [ ]:
# Visualize Finer-CAM on more examples
n_show, shown = 20, 0  # how many images to visualize

for images, labels, names in test_loader:
    for i in [10, 50]:
        x, label = images[i:i+1].to(device), labels[i].item()
        with torch.no_grad():
            pred = resenet50_linearprob_model(x).argmax(1).item()  # the model's actual prediction
        cam, _, refs = finer_cam(resenet50_linearprob_model, x, target=None)  # or refs=<class idx> to pick the reference yourself

        raw = np.clip(x[0].cpu().numpy().transpose(1, 2, 0) * np.array(std) + np.array(mean), 0, 1)
        heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)[..., ::-1] / 255.0
        overlay = 0.4 * heat + 0.6 * raw

        fig, ax = plt.subplots(1, 2, figsize=(8, 4))
        ax[0].imshow(raw);     ax[0].set_title(f"Original (true {label}, pred {pred})"); ax[0].axis("off")
        ax[1].imshow(overlay); ax[1].set_title(f"Finer-CAM: {pred} vs {refs}", fontsize=9); ax[1].axis("off")
        plt.show(); plt.close(fig)

        shown += 1
        if shown >= n_show: break
    if shown >= n_show: break